# Análisis de vuelos
# Scraping y extracción

 En este laboratorio vamos a abordar la extracción de información mediante web scraping, para luego extraer esa información y almacenar en archivo csv.


## Instalación e Importación de librerías

In [1]:
!pip install xlsxwriter
!pip install tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.6 MB/s eta 0:00:00


In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from datetime import datetime, timedelta
import os

## Declaración de constantes

In [3]:
# url = "https://failbondi.fail/?date="

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

months_max_days = { "01": 31, "02": 28, "03": 31, "04": 30, "05": 31, "06": 30, "07": 31, "08": 31, "09": 30, "10": 31, "11": 30, "12": 31 }
month_days = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11","12","13","14","15","16","17","18","19","20", "21", "22", "23", "24", "25", "26", "27", "28", "29", "30", "31"]
FLYBONDI = "FO"
JETSMART = "WJ"

In [4]:
def generate_url(date, company):
    return f"https://failbondi.fail/?date={date}&aerolinea={company}"


## Parámetros de extracción (año y rango de meses)

Usá el formulario de la celda siguiente para elegir el **año** y el **rango de meses** (mes desde / mes hasta) que querés scrapear. Así evitás tener que procesar los 12 meses si solo necesitás algunos.

In [5]:
#@title Parámetros de extracción { display-mode: "form" }
anio = "2025" # @param ["2025","2026"]
company = "JETSMART" # @param ["FLYBONDI","JETSMART"]

mes_desde = 1 #@param {type:"slider", min:1, max:12, step:1}
mes_hasta = 2 #@param {type:"slider", min:1, max:12, step:1}

#company = JETSMART

In [8]:
# Validación de los parámetros ingresados en el formulario
YEAR = anio
ANIOS_HABILITADOS = ["2025", "2026"]

def validar_parametros(anio, mes_desde, mes_hasta):
    errores = []

    # Año: por el momento solo se puede scrapear 2025 o 2026
    if str(anio) not in ANIOS_HABILITADOS:
        errores.append(
            f"El año '{anio}' no es válido. Por el momento solo se puede seleccionar: "
            f"{', '.join(ANIOS_HABILITADOS)}."
        )

    # Meses: enteros entre 1 y 12
    if not (1 <= mes_desde <= 12):
        errores.append(f"'mes_desde' ({mes_desde}) debe estar entre 1 y 12.")
    if not (1 <= mes_hasta <= 12):
        errores.append(f"'mes_hasta' ({mes_hasta}) debe estar entre 1 y 12.")

    # Coherencia del rango
    if 1 <= mes_desde <= 12 and 1 <= mes_hasta <= 12 and mes_desde > mes_hasta:
        errores.append(f"'mes_desde' ({mes_desde}) no puede ser mayor que 'mes_hasta' ({mes_hasta}).")

    if errores:
        raise ValueError("Parámetros inválidos:\n- " + "\n- ".join(errores))

    anio_int = int(anio)
    mes_desde = int(mes_desde)
    mes_hasta = int(mes_hasta)

    # No se puede scrapear más allá del día de ayer (todavía no hay datos del día en curso)
    fecha_limite = datetime.now().date() - timedelta(days=1)

    if anio_int == fecha_limite.year:
        mes_limite = fecha_limite.month

        if mes_desde > mes_limite:
            raise ValueError(
                f"No hay datos disponibles: el rango solicitado empieza en el mes {mes_desde:02d}, "
                f"pero todavía no hay información más allá del {fecha_limite.strftime('%d-%m-%Y')}."
            )

        if mes_hasta > mes_limite:
            print(
                f"Aviso: no se puede hacer scraping de meses posteriores al {fecha_limite.strftime('%d-%m-%Y')}. "
                f"Se ajusta el rango de {mes_desde:02d}-{mes_hasta:02d} a {mes_desde:02d}-{mes_limite:02d}."
            )
            mes_hasta = mes_limite

    return str(anio), mes_desde, mes_hasta


year, mes_desde, mes_hasta = validar_parametros(anio, mes_desde, mes_hasta)

# Filtramos el diccionario original de meses según el rango elegido (ya validado y ajustado)
meses_seleccionados = {
    mes: dias for mes, dias in months_max_days.items()
    if mes_desde <= int(mes) <= mes_hasta
}

print(f"Año seleccionado: {year}")
print(f"Rango de meses: {mes_desde:02d} a {mes_hasta:02d}")
print(f"Meses a procesar: {list(meses_seleccionados.keys())}")


Año seleccionado: 2025
Rango de meses: 01 a 02
Meses a procesar: ['01', '02']


## Funciones reutilizables

In [9]:
def get_html_from_url(url, headers):
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, "html.parser")
        return soup

    return None

def scraping_vuelos(html, date, company):
    """
    Extracts flight data from an HTML table and structures it into a DataFrame.

    Args:
        html (bs4.BeautifulSoup): The BeautifulSoup object containing the parsed HTML.
        date (str or datetime): The reference date to be associated with the scraped data.

    Returns:
        pd.DataFrame: A cleaned DataFrame containing flight records, including
                      standardized dates and month extraction for grouping.
    """
    # 1. Extract headers
    headers = [th.text.strip() for th in html.find('thead').find_all('th')]

    # 2. Extraer filas
    rows = []
    table_body = html.find('tbody')
    for tr in table_body.find_all('tr'):
        cells = [td.text.strip() for td in tr.find_all('td')]
        rows.append(cells)

    # 3. Crear DataFrame
    df = pd.DataFrame(rows, columns=headers)
    df['fecha'] = date
    df['fecha'] = pd.to_datetime(df['fecha'])
    df['mes'] = df['fecha'].dt.month
    df['empresa'] = company

    return df


def get_report_by_month(year_month, max_days, company):
    lista_dfs = []
    for i in range(max_days):
        date = f"{year_month}-{month_days[i]}"
        url_link = generate_url(date, company=company)

        time.sleep(random.uniform(1.5, 3.5))

        main_content = get_html_from_url(url_link, headers)

        try:
            df_iteracion = scraping_vuelos(main_content, date, company)
            if not df_iteracion.empty:
                lista_dfs.append(df_iteracion)
        except Exception as e:
            print(f"Error en fecha {date}: {e}")
            time.sleep(10)

    if not lista_dfs: return pd.DataFrame()

    df_month = pd.concat(lista_dfs, ignore_index=True)
    print(f"[{year_month}] - Filas obtenidas: {len(df_month)}")
    return df_month


In [11]:
lista_dfs = []

inicio_peticion_total = time.time()

for month, max_days in meses_seleccionados.items():
    year_month = f"{YEAR}-{month}"
    print(f"Iniciando extracción de: {year_month}")

    inicio_peticion = time.time()

    # lista_dfs.append(get_report_by_month(year_month, max_days))
    df_mensual = get_report_by_month(year_month, max_days, company=company)

    fin_peticion = time.time()
    duracion = fin_peticion - inicio_peticion
    hora_actual = datetime.now().strftime('%H:%M:%S')

    filename = f"raw_vuelos_{company}_{year_month}.csv"
    df_mensual.to_csv(filename, index=False)
    print(f"Archivo {filename} guardado con éxito.")

    # Descanso largo entre meses para "enfriar" la IP
    time.sleep(random.uniform(10, 20))

    print(f"[{hora_actual}] Finalizado: {year_month} | Tiempo: {duracion:.2f}s")

fin_peticion_total = time.time()
duracion_total = fin_peticion_total - inicio_peticion_total

print("Duración total del proceso: ", duracion_total)


Iniciando extracción de: 2025-01
[2025-01] - Filas obtenidas: 2306
Archivo raw_vuelos_JETSMART_2025-01.csv guardado con éxito.
[02:33:49] Finalizado: 2025-01 | Tiempo: 93.81s
Iniciando extracción de: 2025-02
[2025-02] - Filas obtenidas: 1968
Archivo raw_vuelos_JETSMART_2025-02.csv guardado con éxito.
[02:35:33] Finalizado: 2025-02 | Tiempo: 85.92s
Duración total del proceso:  208.69609427452087


In [12]:
import glob
import os


def export_and_unify_files(folder_path, company, format="parquet", prefijo=None):
    # 1. Buscar todos los archivos mensuales'
    patron = "*.csv"

    if prefijo:
        patron = f"{prefijo}*.csv"

    archivos = glob.glob(os.path.join(folder_path, patron))
    archivos.sort() # Para mantener el orden cronológico

    lista_dfs = []
    for archivo in archivos:
        print(f"Leyendo: {archivo}")
        df = pd.read_csv(archivo)
        lista_dfs.append(df)

    # 2. Unir todos (pandas ignora los headers repetidos y crea un solo esquema)
    df_final = pd.concat(lista_dfs, ignore_index=True)

    nombre_archivo = f"vuelos_anual_{company}_consolidado_{YEAR}"
    # 3. Exportar según formato
    if format == "csv":
        df_final.to_csv(nombre_archivo + ".csv", index=False)
    elif format == "parquet":
        # Requiere: pip install pyarrow fastparquet
        df_final.to_parquet(nombre_archivo + ".parquet", index=False)
    elif format == "excel":
        df_final.to_excel(nombre_archivo + ".xlsx", index=False)

    print(f"Consolidación exitosa. Filas totales: {len(df_final)}")
    return df_final


In [13]:
%pwd

'/content'

In [14]:
export_and_unify_files('/content', company)
export_and_unify_files('/content', company, 'csv')


Leyendo: /content/raw_vuelos_JETSMART_2025-01.csv
Leyendo: /content/raw_vuelos_JETSMART_2025-02.csv
Consolidación exitosa. Filas totales: 4274
Leyendo: /content/raw_vuelos_JETSMART_2025-01.csv
Leyendo: /content/raw_vuelos_JETSMART_2025-02.csv
Consolidación exitosa. Filas totales: 4274


,Vuelo,Ruta,Hora Programada,Hora Real,Demora en despegar,fecha,mes,empresa
0,FO 5912,Aeroparque → Rio de Janeiro,13:05,NaN,Cancelado,2025-01-01,1,JETSMART
1,FO 5237,Bariloche → Aeroparque,07:50,19:22,11hs 32min tarde,2025-01-01,1,JETSMART
2,FO 5236,Ezeiza → Bariloche,05:00,15:16,10hs 16min tarde,2025-01-01,1,JETSMART
3,FO 5027,Córdoba → Ezeiza,20:40,02:18 +1,5hs 38min tarde,2025-01-01,1,JETSMART
4,FO 5056,Aeroparque → Mendoza,16:00,21:27,5hs 27min tarde,2025-01-01,1,JETSMART
...,...,...,...,...,...,...,...,...
4269,FO 5952,Ezeiza → Florianopolis,18:00,18:19,19min tarde,2025-02-28,2,JETSMART
4270,FO 5320,Aeroparque → Puerto Madryn,07:20,07:35,15min tarde,2025-02-28,2,JETSMART
4271,FO 5241,Bariloche → Ezeiza,11:05,11:20,15min tarde,2025-02-28,2,JETSMART
4272,FO 5900,Ezeiza → Rio de Janeiro,06:20,06:31,11min tarde,2025-02-28,2,JETSMART


Este notebook finaliza con el archivo resultante de todos los vuelos que hay en la página de todo el año 2025.

Consideraciones al realizar web scraping, para evitar que la IP sea baneada se considero agregar tiempos de espera ```sleep```, durante las peticiones por fechas y en caso que surja algún error se agrega tiempo extra.

